In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from transformers import BitsAndBytesConfig
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

model_id = "large-traversaal/Alif-1.0-8B-Instruct"

# 4-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load tokenizer and model in 4-bit
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=quantization_config, device_map="auto"
)

# Create text generation pipeline
chatbot = pipeline(
    "text-generation", model=model, tokenizer=tokenizer, device_map="auto"
)


# Function to chat
def chat(message):
    response = chatbot(
        message,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
    )
    return response[0]["generated_text"]


# Example chat
user_input = "شہر کراچی کی کیا اہمیت ہے؟"
bot_response = chat(user_input)

print(bot_response)

In [ ]:
dataset = load_dataset("large-traversaal/mgsm_urdu_final", split="test")
results = []

for row in tqdm(dataset):
    response = chat(row["urdu_question"])  # Your model response

    results.append(
        {
            "question": row["urdu_question"],
            "answer": row["answer_number"],
            "response": response,
        }
    )

df = pd.DataFrame(results)
df.to_csv("alif-instruct-response.csv", index=False)

### LLM-as-a-Judge For Evaluation


In [ ]:
df = pd.read_csv("alif-instruct-response.csv")

validation_results = []

model_id = "openai/gpt-oss-20b"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto",
)

for _, row in tqdm(df.iterrows()):

    prompt = f"""You are an answer validator for Urdu math reasoning problems.
    
    Input:
    
    * Question: {row['question']}
    * Solution: {row['response']}
    * Expected Answer: {row['answer']}
    
    Task:
    
    1. Read the Urdu math problem and the provided solution.
    2. Determine the final answer produced by the solution.
    3. Compare it with the Expected Answer.
    4. Treat numerically equivalent values as equal:
    
       * 2 = 2.0 = 2.00
       * 5 = 5.000
       * Ignore insignificant trailing zeros in decimals.
    5. If both answers represent the same numeric value, return:
       True
    6. Otherwise return:
       False
    
    Rules:
    
    * Return ONLY one word: True or False.
    * Do not provide explanations, reasoning, calculations, or extra text.
    * Comparison should be based on the final answer only.
    * For numeric answers, compare by value, not string format.
    * If the solution's final answer cannot be determined with confidence, return False.
     """

    messages = [{"role": "user", "content": prompt}]
    outputs = pipe(messages, max_new_tokens=1000)

    result = (
        True
        if "True.assistantfinalTrue" in outputs[0]["generated_text"][-1]["content"]
        else False
    )
    print(outputs[0]["generated_text"][-1]["content"])
    print(result)
    print("\n\n")
    validation_results.append(result)

df["is_correct"] = validation_results

df.to_csv("alif-instruct_correct.csv", index=False)

print(df.head())